In [ ]:
%%bash
# Cell 1: Environment & Advanced Dependencies
git clone https://github.com/kohya-ss/sd-scripts.git /kaggle/working/sd-scripts
cd /kaggle/working/sd-scripts

pip install --upgrade pip
pip install -r requirements.txt
pip install xformers bitsandbytes accelerate prodigyopt

accelerate config default

In [ ]:
# Cell 2: Multi-Angle Dataset Synthesis & Pipeline
import os
import glob
from PIL import Image

input_dir = "/kaggle/input/datasets/anindoakb/guinevere-lora"
output_dir = "/kaggle/working/train_guinevere/img/40_guinevere_mlbb"
os.makedirs(output_dir, exist_ok=True)

image_paths = glob.glob(os.path.join(input_dir, "*.*"))

for idx, img_path in enumerate(image_paths):
    try:
        img = Image.open(img_path).convert("RGB")
        
        # Save as high-quality JPG to prevent tensor errors
        base_out = os.path.join(output_dir, f"img_{idx:04d}.jpg")
        img.save(base_out, "JPEG", quality=100)
        
        # Augment with horizontal flip for 360-degree pose learning
        img_flip = img.transpose(Image.FLIP_LEFT_RIGHT)
        flip_out = os.path.join(output_dir, f"img_{idx:04d}_aug_flip.jpg")
        img_flip.save(flip_out, "JPEG", quality=100)
    except Exception as e:
        print(f"Skipping {img_path} due to error: {e}")

print(f"Dataset synthesis complete. Images: {len(os.listdir(output_dir))}")

In [ ]:
# Cell 3: 20 HD Anime Sample Prompts (Different Poses)
import os

prompt_file = "/kaggle/working/prompt.txt"

# 20 completely different poses and angles
prompts = [
    "guinevere_mlbb, 1girl, fighting stance, glowing magic, intense gaze, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, jumping mid-air, dynamic action scene, flowing hair, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, sitting elegantly on a throne, regal posture, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, looking over shoulder, seductive smile, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, crouching low, stealthy, ready to pounce, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, casting magic, ethereal glow, magical circle, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, running towards the camera, breathless, action blur, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, lying on the grass, peaceful expression, looking at sky, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, back to camera, turning head, flowing dress, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, mid-kick, martial arts pose, dynamic composition, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, floating in water, ethereal dress, serene face, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, kneeling in prayer, hands clasped, divine light, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, pointing finger forward, commanding presence, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, leaning against a wall, casual yet elegant, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, doing a pirouette, ballet pose, twirling skirt, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, upside down, falling through the sky, wind in hair, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, drawing a sword, intense combat readiness, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, hugging knees to chest, vulnerable, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, reaching hand out, inviting gesture, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face",
    "guinevere_mlbb, 1girl, profile view, side angle, gazing into the distance, masterpiece, best quality, ultra-detailed anime illustration, 8k resolution, highly detailed face"
]

# Kohya's sample prompt format flags appended to the end of every line
flags = "--w 768 --h 1024 --l 7.5 --s 30"

with open(prompt_file, "w", encoding="utf-8") as f:
    for prompt in prompts:
        f.write(f"{prompt} {flags}\n")

print(f"Generated 20 sample prompts in {prompt_file}")

In [ ]:
%%bash
# Cell 4: Maximum Capability Training Execution
cd /kaggle/working/sd-scripts

# Train with Prodigy optimizer, min-SNR gamma, and multires noise for optimal HD Anime stylization
accelerate launch train_network.py \
  --pretrained_model_name_or_path="andite/anything-v4.0" \
  --train_data_dir="/kaggle/working/train_guinevere/img" \
  --output_dir="/kaggle/working/output_guinevere" \
  --network_module="networks.lora" \
  --network_dim=256 \
  --network_alpha=128 \
  --optimizer_type="Prodigy" \
  --learning_rate=1.0 \
  --unet_lr=1.0 \
  --text_encoder_lr=1.0 \
  --lr_scheduler="cosine" \
  --optimizer_args weight_decay=0.01 d_coef=2 use_bias_correction=True safeguard_warmup=True \
  --min_snr_gamma=5.0 \
  --noise_offset=0.1 \
  --multires_noise_iterations=6 \
  --multires_noise_discount=0.3 \
  --sample_prompts="/kaggle/working/prompt.txt" \
  --sample_every_n_epochs=5 \
  --mixed_precision="bf16" \
  --save_precision="bf16" \
  --xformers \
  --max_train_epochs=30

In [ ]:
# Cell 5: Exporting
import shutil
import os

# Zip output_guinevere folder to capture both .safetensors and sample/ images
output_folder = "/kaggle/working/output_guinevere"
archive_name = "/kaggle/working/guinevere_lora_outputs"

if os.path.exists(output_folder):
    shutil.make_archive(archive_name, 'zip', output_folder)
    print(f"Export complete. Archive saved to {archive_name}.zip")
else:
    print(f"Folder {output_folder} not found. Ensure training finished successfully.")